# Part V: Numerical methods: solutions to the code exercises

Worked solutions to the code exercises from [*Matrices in Practice*](https://trgoodale.github.io/matrix-book/appendices/solutions-5.html), by Tom Goodale.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trgoodale/matrix-book/blob/main/code/python/solutions-part-5.ipynb)

Run the cells in order, from the top: later cells use variables set in earlier ones. Try each exercise yourself first. Change the numbers and run them again to see what happens. The code needs NumPy, SciPy and Matplotlib (and pandas in a few chapters), all of which Colab already has.

Code licensed under the [MIT licence](https://trgoodale.github.io/matrix-book/LICENSE.md).

## Chapter 24: Floating point and conditioning

### Exercise 7

For the Hilbert matrices with $n = 2, 3, \dots, 14$, compute $\kappa_2(\mathbf{H}_n)$ and the error in solving $\mathbf{H}_n\hat{\mathbf{x}} = \mathbf{H}_n\mathbf{1}$. Plot both on a logarithmic scale against $n$, and check the rule of thumb. Roughly how fast does $\kappa$ grow with $n$?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import hilbert

ns = range(2, 15)
conds, errors = [], []
for n in ns:
    H = hilbert(n)
    xhat = np.linalg.solve(H, H @ np.ones(n))
    conds.append(np.linalg.cond(H))
    errors.append(np.max(np.abs(xhat - 1)))

plt.figure(figsize=(6, 3.5))
plt.semilogy(ns, conds, "o-", label="condition number")
plt.semilogy(ns, errors, "s-", label="error in solution")
plt.semilogy(ns, np.array(conds) * 1.1e-16, "--", label="condition number × ε/2")
plt.xlabel("n")
plt.legend()
plt.show()

The error follows the line $\kappa\varepsilon/2$ closely, as the rule of thumb predicts, until $\kappa$ passes $10^{16}$ and the solution has no correct digits. The condition number grows by a factor of about 30 each time $n$ increases by 1, so roughly like $10^{1.5n}$. (Beyond $n \approx 13$, the computed condition number itself is no longer accurate, which is why it stops growing.)

## Chapter 25: Pivoting

### Exercise 7

Build Wilkinson's matrix for $n = 10$, $30$ and $60$, choose a random $\mathbf{x}$, set $\mathbf{b} = \mathbf{W}\mathbf{x}$, and solve with `np.linalg.solve` or backslash. Compare the relative error with the condition number of $\mathbf{W}$, and with the largest entry of $\mathbf{U}$. What do you conclude?

In [ ]:
import numpy as np
import scipy.linalg

def wilkinson(n):
    W = np.eye(n) - np.tril(np.ones((n, n)), -1)
    W[:, -1] = 1
    return W

rng = np.random.default_rng(0)
for n in [10, 30, 60]:
    W = wilkinson(n)
    x = rng.standard_normal(n)
    xhat = np.linalg.solve(W, W @ x)
    P, L, U = scipy.linalg.lu(W)
    print(f"n = {n}: cond = {np.linalg.cond(W):.1f}, largest entry of U = {np.abs(U).max():.1e}, "
          f"relative error = {np.linalg.norm(xhat - x, np.inf) / np.linalg.norm(x, np.inf):.1e}")

The condition number stays small (about 27 for $n = 60$), so the problem is well conditioned, yet the error grows from about $10^{-14}$ to tens of percent as the largest entry of $\mathbf{U}$ grows like $2^{n-1}$, to about $6 \times 10^{17}$. Partial pivoting is not backward stable for this matrix: the growth factor magnifies rounding errors exactly as large multipliers do without pivoting. Such matrices are extremely rare in practice, but they show that the guarantee is not absolute.

## Chapter 26: Cost, banded and sparse systems

### Exercise 7

Time the solution of the tridiagonal system in the code above for $n = 500$, $1000$, $2000$ and $4000$, using (a) a dense matrix and `np.linalg.solve` or backslash on a full matrix, (b) the sparse solver. On a log–log plot, estimate the power of $n$ that each time grows like.

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve
import time

ns = [500, 1000, 2000, 4000]
dense_t, sparse_t = [], []
for n in ns:
    A = sp.diags([-1.0, 2.0, -1.0], [-1, 0, 1], shape=(n, n), format="csc")
    b = np.ones(n)
    Ad = A.toarray()
    start = time.perf_counter(); np.linalg.solve(Ad, b); dense_t.append(time.perf_counter() - start)
    start = time.perf_counter(); spsolve(A, b); sparse_t.append(time.perf_counter() - start)

for name, t in [("dense", dense_t), ("sparse", sparse_t)]:
    slope = np.polyfit(np.log(ns), np.log(t), 1)[0]
    print(f"{name}: times {[f'{x:.4f}' for x in t]}, grows like n^{slope:.1f}")

Timings vary from run to run and between computers, especially for small, fast cases, but the dense times grow roughly like $n^3$ (often a little less for these sizes, because fast libraries run faster on larger matrices). The sparse times are so small, around a millisecond, that fixed overheads dominate; trying $n$ up to $10^6$ shows them growing like $n$. MATLAB's backslash notices that a full matrix is tridiagonal and uses a banded solver even on `Ad`, so its dense times may grow much more slowly; NumPy's `solve` always does dense elimination.

## Chapter 27: Iterative methods

### Exercise 7

Modify the code above to compare Jacobi, Gauss–Seidel, best SOR and CG for $n = 25, 50, 100$ and $200$. How does the number of steps grow with $n$ for each method? Plot the residual against the step number for $n = 100$ on a logarithmic scale.

In [ ]:
import numpy as np
from scipy.linalg import solve_triangular
from scipy.sparse.linalg import cg

def count_steps(n):
    A = 2 * np.eye(n) - np.eye(n, k=1) - np.eye(n, k=-1)
    b = np.ones(n) / (n + 1)**2
    D, L, U = np.diag(np.diag(A)), np.tril(A, -1), np.triu(A, 1)
    rho = np.cos(np.pi / (n + 1))
    w = 2 / (1 + np.sqrt(1 - rho**2))
    out = {}
    for name, (M, N) in {"Jacobi": (D, L + U), "Gauss-Seidel": (D + L, U),
                         "SOR": (D / w + L, U + (1 - 1 / w) * D)}.items():
        if name != "SOR" and n > 100:
            out[name] = None            # too slow to run here
            continue
        x, k = np.zeros(n), 0
        while np.linalg.norm(b - A @ x) >= 1e-6 * np.linalg.norm(b):
            # M is diagonal or lower triangular, so use a triangular solve:
            # it is much cheaper than a general solve, run tens of thousands of times
            x = solve_triangular(M, b - N @ x, lower=True)
            k += 1
        out[name] = k
    steps = []
    cg(A, b, rtol=1e-6, callback=lambda xk: steps.append(1))
    out["CG"] = len(steps)
    return out

for n in [25, 50, 100, 200]:
    print(n, count_steps(n))

Each doubling of $n$ roughly multiplies the Jacobi and Gauss–Seidel counts by 4, so they grow like $n^2$, because $1 - \rho(\mathbf{T}_J) = 1 - \cos\bigl(\pi/(n+1)\bigr) \approx \pi^2/\bigl(2(n + 1)^2\bigr)$. (For $n = 200$, Jacobi would need about $110\,000$ steps and Gauss–Seidel half as many, so the code skips them.) Best SOR and CG grow like $n$: SOR needs about $3n$ steps here, and CG exactly $n/2$. That is because $\mathbf{b}$ is symmetric about the middle of the rod, so it is a combination of only half of the eigenvectors of $\mathbf{A}$, and in exact arithmetic CG needs at most one step for each eigenvector involved. The MATLAB version follows the chapter's code in the same way, with `pcg` for CG. Plotting the residual against the step number on a log scale shows straight lines for the stationary methods, whose slopes are $\log_{10}\rho$, and a curve that drops away steeply for CG.

## Chapter 28: Newton's method for nonlinear systems

### Exercise 7

Write a Newton solver that approximates the Jacobian by finite differences, and use it to find all the intersections of $x^2 + 4y^2 = 4$ and $y = \sin(2x)$. Try several starting points, and compare the number of steps needed with the exact-Jacobian version.

In [ ]:
import numpy as np

def f(v):
    x, y = v
    return np.array([x**2 + 4 * y**2 - 4, y - np.sin(2 * x)])

def fd_jacobian(f, x, h=1e-7):
    fx = f(x)
    J = np.zeros((len(fx), len(x)))
    for j in range(len(x)):
        step = h * max(1, abs(x[j]))
        e = np.zeros(len(x)); e[j] = step
        J[:, j] = (f(x + e) - fx) / step
    return J

def newton(f, x, tol=1e-12, maxit=50):
    x = np.array(x, dtype=float)
    for k in range(1, maxit + 1):
        dx = np.linalg.solve(fd_jacobian(f, x), -f(x))
        x = x + dx
        if np.linalg.norm(dx) < tol:
            return x, k
    return x, None

for start in [(-2, 0.5), (-1, -1), (-0.5, -1), (0.5, 1), (1, 1), (2, -0.5)]:
    root, steps = newton(f, start)
    print(start, np.round(root, 6), steps, "steps")

There are six intersections, at about $(\pm 0.626, \pm 0.950)$, $(\pm 1.067, \pm 0.846)$ and $(\mp 1.798, \pm 0.438)$ (taking the signs together, since the curves are symmetric under $(x, y) \to (-x, -y)$). From these starts, the finite-difference version needs 5 or 6 steps, exactly the same as with the exact Jacobian: the small errors in the approximate Jacobian hardly slow convergence. Starting at $(0, 0)$ fails, because the Jacobian $\begin{pmatrix}2x & 8y \\ -2\cos 2x & 1\end{pmatrix}$ is singular there.

## Chapter 29: Eigenvalue algorithms

### Exercise 7

Build the $n \times n$ matrix with 2 on the diagonal and $-1$ beside it, for $n = 1000$, as a sparse matrix. Use `eigsh` (Python) or `eigs` (MATLAB) to find its 5 smallest eigenvalues, and compare them with the exact values $2 - 2\cos\bigl(k\pi/(n + 1)\bigr)$, $k = 1, \dots, 5$. Then try the power method on it: how many steps would it need to separate the two largest eigenvalues?

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh

n = 1000
T = sp.diags([-1.0, 2.0, -1.0], [-1, 0, 1], shape=(n, n), format="csc")
values = eigsh(T, k=5, sigma=0, return_eigenvectors=False)   # nearest 0
exact = 2 - 2 * np.cos(np.arange(1, 6) * np.pi / (n + 1))
print(np.sort(values))
print(exact)

l1 = 2 - 2 * np.cos(n * np.pi / (n + 1))          # the two largest
l2 = 2 - 2 * np.cos((n - 1) * np.pi / (n + 1))
print(l2 / l1, 1 / -np.log10(l2 / l1))

The five computed eigenvalues agree with the formula to about 12 significant digits. For the power method, the two largest eigenvalues are about $3.99999015$ and $3.99996060$, with ratio $0.9999926$, so each digit would need about $311\,000$ steps: hopeless. The shift-and-invert Lanczos method behind `eigsh` and `eigs` needs only a few dozen steps.

## Chapter 30: QR factorisation and stable least squares

### Exercise 7

Implement classical and modified Gram–Schmidt. For the matrices with columns $1, t, \dots, t^{n-1}$ at 50 points in $[0, 1]$, with $n = 4, 6, 8, 10, 12$, compute $\lVert\mathbf{Q}^{\mathsf{T}}\mathbf{Q} - \mathbf{I}\rVert$ for both versions and for `np.linalg.qr` or `qr`, and plot the results against $\kappa(\mathbf{A})$ on log scales. What do you see?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def classical_gs(A):
    Q = np.zeros(A.shape)
    for j in range(A.shape[1]):
        v = A[:, j] - Q[:, :j] @ (Q[:, :j].T @ A[:, j])   # project the original column
        Q[:, j] = v / np.linalg.norm(v)
    return Q

def modified_gs(A):
    Q = A.astype(float).copy()
    for j in range(A.shape[1]):
        Q[:, j] /= np.linalg.norm(Q[:, j])
        for k in range(j + 1, A.shape[1]):              # update the remaining columns now
            Q[:, k] -= (Q[:, j] @ Q[:, k]) * Q[:, j]
    return Q

t = np.linspace(0, 1, 50)
conds, loss = [], {"classical": [], "modified": [], "Householder": []}
for n in [4, 6, 8, 10, 12]:
    A = np.vander(t, n, increasing=True)
    conds.append(np.linalg.cond(A))
    for name, Q in [("classical", classical_gs(A)), ("modified", modified_gs(A)),
                    ("Householder", np.linalg.qr(A)[0])]:
        loss[name].append(np.linalg.norm(Q.T @ Q - np.eye(n)))

plt.figure(figsize=(6, 3.5))
for name, values in loss.items():
    plt.loglog(conds, values, "o-", label=name)
plt.xlabel("condition number of A")
plt.ylabel("‖QᵀQ − I‖")
plt.legend()
plt.show()

Householder's $\mathbf{Q}$ stays orthogonal to about $10^{-15}$ whatever the conditioning. Modified Gram–Schmidt loses orthogonality in proportion to $\kappa(\mathbf{A})\varepsilon$, and classical Gram–Schmidt in proportion to $\kappa(\mathbf{A})^2\varepsilon$, so on log scales they are straight lines of slopes 1 and 2, until classical reaches order 1 and its "$\mathbf{Q}$" is useless.

## Part V: Numerical methods review

### Exercise 9

Run the Jacobi method on $\mathbf{A}\mathbf{x} = \mathbf{b}$, with $\mathbf{A}$ from exercise 4 and $\mathbf{b} = (6, 12)^{\mathsf{T}}$, starting from $\mathbf{0}$. Print the ratio of successive errors $\lVert\mathbf{x}_{k+1} - \mathbf{x}\rVert / \lVert\mathbf{x}_k - \mathbf{x}\rVert$, and compare it with the spectral radius. Repeat with Gauss–Seidel.

*Uses [the chapter](https://trgoodale.github.io/matrix-book/part-5/27-iterative-methods.html).*

In [ ]:
import numpy as np

A = np.array([[4.0, 1], [2, 5]])
b = np.array([6.0, 12])
x_true = np.linalg.solve(A, b)
D = np.diag(np.diag(A))
L = np.tril(A, -1)

for name, M in (("Jacobi", D), ("Gauss-Seidel", D + L)):
    N = M - A                          # A = M - N, iterate M x_new = N x + b
    rho = max(abs(np.linalg.eigvals(np.linalg.solve(M, N))))
    print(f"{name}, spectral radius {rho:.4f}")
    x = np.zeros(2)
    err = np.linalg.norm(x - x_true)
    for k in range(1, 7):
        x = np.linalg.solve(M, N @ x + b)
        new_err = np.linalg.norm(x - x_true)
        print(f"  step {k}: error {new_err:.2e}, ratio {new_err / err:.4f}")
        err = new_err

The solution is $\mathbf{x} = (1, 2)$. For Gauss–Seidel the ratio settles at exactly $\rho = 0.1$. For Jacobi it alternates between about 0.29 and 0.35, because its two eigenvalues, $\pm 0.316$, have the same size; over two steps the error shrinks by exactly $0.316^2 = 0.1$, so on average by $\rho$ per step.